# Phase 1: Data Prep & EDA (MVTec AD Pill)

Run on Google Colab, top to bottom. No GPU is needed for this notebook.

Rule: test images here are for understanding the data only. Do not use them to choose
hyperparameters or thresholds.

## 1. Save the dataset script

This is a copy of `src/prepare_dataset.py`.

In [ ]:
%%writefile prepare_dataset.py
"""Validate the MVTec AD Pill folder and write a reproducible train/validation split.

Uses only the standard library so it runs anywhere, including Colab.

    python src/prepare_dataset.py --root /path/to/mvtec --out splits/pill_seed42.json
"""
import argparse
import json
import random
import sys
from pathlib import Path

EXPECTED_TRAIN_GOOD = 267
EXPECTED_TEST_GOOD = 26
EXPECTED_TEST_DEFECT = 141
EXPECTED_DEFECTS = {
    "color",
    "combined",
    "contamination",
    "crack",
    "faulty_imprint",
    "pill_type",
    "scratch",
}


class DatasetError(Exception):
    """The dataset folder does not match the expected structure."""


def find_category_root(root, category="pill"):
    """Return the category folder, whether root is that folder or a parent of it."""
    root = Path(root)
    if (root / "train").is_dir() and (root / "test").is_dir():
        return root
    for candidate in sorted(root.rglob(category)):
        if (candidate / "train").is_dir() and (candidate / "test").is_dir():
            return candidate
    raise DatasetError(f"No '{category}' folder with train/ and test/ under {root}")


def _pngs(folder):
    return sorted(p.name for p in folder.glob("*.png"))


def inspect_dataset(pill_root):
    """Count images and check every defect image has a mask. Raises DatasetError."""
    pill_root = Path(pill_root)
    problems = []

    train_good = _pngs(pill_root / "train" / "good")
    test_good = _pngs(pill_root / "test" / "good")
    test_dirs = {p.name for p in (pill_root / "test").iterdir() if p.is_dir()}
    defects = sorted(test_dirs - {"good"})

    if set(defects) != EXPECTED_DEFECTS:
        problems.append(
            f"defect folders differ: missing {sorted(EXPECTED_DEFECTS - set(defects))}, "
            f"unexpected {sorted(set(defects) - EXPECTED_DEFECTS)}"
        )

    defect_counts = {}
    for defect in defects:
        images = _pngs(pill_root / "test" / defect)
        defect_counts[defect] = len(images)
        mask_dir = pill_root / "ground_truth" / defect
        masks = set(_pngs(mask_dir)) if mask_dir.is_dir() else set()
        expected_masks = {f"{Path(name).stem}_mask.png" for name in images}
        missing = sorted(expected_masks - masks)
        if missing:
            problems.append(f"{defect}: {len(missing)} missing masks, e.g. {missing[0]}")

    total_defect = sum(defect_counts.values())
    for label, actual, expected in (
        ("train/good", len(train_good), EXPECTED_TRAIN_GOOD),
        ("test/good", len(test_good), EXPECTED_TEST_GOOD),
        ("test defects", total_defect, EXPECTED_TEST_DEFECT),
    ):
        if actual != expected:
            problems.append(f"{label}: expected {expected} images, found {actual}")

    if problems:
        raise DatasetError("; ".join(problems))

    return {
        "train_good": len(train_good),
        "test_good": len(test_good),
        "test_defect": total_defect,
        "defect_counts": defect_counts,
    }


def make_split(pill_root, val_ratio=0.15, seed=42):
    """Split train/good only. The test set is never touched."""
    if not 0 < val_ratio < 1:
        raise ValueError("val_ratio must be between 0 and 1")
    names = _pngs(Path(pill_root) / "train" / "good")
    random.Random(seed).shuffle(names)
    n_val = round(len(names) * val_ratio)
    return {
        "seed": seed,
        "val_ratio": val_ratio,
        "train": sorted(names[n_val:]),
        "validation": sorted(names[:n_val]),
    }


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--root", required=True, help="pill folder or a parent of it")
    parser.add_argument("--out", default="splits/pill_seed42.json")
    parser.add_argument("--val-ratio", type=float, default=0.15)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args(argv)

    try:
        pill_root = find_category_root(args.root)
        summary = inspect_dataset(pill_root)
    except DatasetError as error:
        print(f"Dataset check failed: {error}", file=sys.stderr)
        return 1

    split = make_split(pill_root, args.val_ratio, args.seed)
    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(split, indent=2) + "\n")

    print(json.dumps(summary, indent=2))
    print(f"train={len(split['train'])} validation={len(split['validation'])} -> {out}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

## 2. Download the dataset

This downloads the full MVTec AD dataset (several GB). If Kaggle asks you to sign in,
add your own Kaggle credentials in Colab.

In [ ]:
%pip install -q kagglehub
import kagglehub

DATA_ROOT = kagglehub.dataset_download("ipythonx/mvtec-ad")
print(DATA_ROOT)

## 3. Validate the folder and create the train/validation split

In [ ]:
import json
from pathlib import Path

import prepare_dataset

SPLIT_PATH = Path("splits/pill_seed42.json")
assert prepare_dataset.main(["--root", str(DATA_ROOT), "--out", str(SPLIT_PATH)]) == 0

PILL = prepare_dataset.find_category_root(DATA_ROOT)
SPLIT = json.loads(SPLIT_PATH.read_text())

## 4. Image size, mode and count per group

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image

rows = []
for path in sorted(PILL.glob("*/*/*.png")):
    if path.parts[-3] == "ground_truth":
        continue
    with Image.open(path) as image:
        gray = np.asarray(image.convert("L"), dtype=np.float32)
        rows.append({
            "subset": path.parts[-3],
            "group": path.parts[-2],
            "file": path.name,
            "width": image.width,
            "height": image.height,
            "mode": image.mode,
            "brightness": gray.mean(),
        })
images = pd.DataFrame(rows)
images.groupby(["subset", "group", "width", "height", "mode"]).size().rename("images")

## 5. Sample images with defect masks

In [ ]:
import matplotlib.pyplot as plt

defects = sorted(p.name for p in (PILL / "test").iterdir() if p.is_dir() and p.name != "good")
fig, axes = plt.subplots(2, len(defects), figsize=(2.4 * len(defects), 5))
for column, defect in enumerate(defects):
    path = sorted((PILL / "test" / defect).glob("*.png"))[0]
    mask = PILL / "ground_truth" / defect / f"{path.stem}_mask.png"
    axes[0, column].imshow(Image.open(path).convert("RGB"))
    axes[0, column].set_title(defect, fontsize=9)
    axes[1, column].imshow(Image.open(mask).convert("L"), cmap="gray")
for axis in axes.ravel():
    axis.axis("off")
plt.tight_layout()
plt.show()

## 6. Defect size by type

Small defects are usually harder to localize.

In [ ]:
rows = []
for mask_path in sorted((PILL / "ground_truth").glob("*/*_mask.png")):
    mask = np.asarray(Image.open(mask_path).convert("L")) > 0
    rows.append({"defect": mask_path.parent.name, "area_percent": 100 * mask.mean()})
masks = pd.DataFrame(rows)
masks.groupby("defect")["area_percent"].describe().round(3)

## 7. Check that train and validation look alike

In [ ]:
train_good = images[(images["subset"] == "train") & (images["group"] == "good")]
split_of = {name: "train" for name in SPLIT["train"]}
split_of.update({name: "validation" for name in SPLIT["validation"]})
train_good = train_good.assign(split=train_good["file"].map(split_of))
assert train_good["split"].notna().all()
train_good.groupby("split")["brightness"].describe().round(2)

## 8. Save the split

Add this file to `splits/` in the repo so later phases use the same split.

In [ ]:
from google.colab import files

files.download(str(SPLIT_PATH))